# Word BoW and TF-IDF with a pairwise classifier

Each text of a pair becomes a word BoW or TF-IDF vector; the two vectors of a pair are combined
into one feature vector (their absolute difference or their element-wise product) and one of six
classifiers decides whether the texts were written by the same author. The vectorizer and the
classifier hyperparameters are tuned together with Optuna on the validation pairs, the final model
is trained on the training and validation pairs and evaluated once on the test pairs.

One run tunes ONE classifier for one representation and one combination. A restarted run continues
where it stopped: the study lives in a SQLite file and a finished run is not repeated. A study is
never continued with other settings than the ones it was started with.

In [ ]:
SMOKE = False  # True - the first rows of every data set, files of their own
N_TRIALS = 30  # finished trials of the study (complete; nothing is pruned)
SEED = 42  # seed of the sampler; the classifiers keep classifiers.SEED
RUN_TAG = ""  # label in the file names of a repeated run
REPRESENTATION = "tfidf"  # "tfidf" or "bow"
COMBINATION = "absolute_difference"  # "absolute_difference" or "element-wise"
CLASSIFIER = "logistic_regression"  # one of classifiers.CLASSIFIER_NAMES
N_JOBS = 1  # threads of kNN, random forest and XGBoost - no effect on the results

Standard libraries

In [ ]:
import logging
import re
import tempfile
import time
from functools import partial
from itertools import islice
from pathlib import Path

Models, data, plots and Optuna

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import optuna
import seaborn as sns
from numpy.core._multiarray_umath import __cpu_dispatch__, __cpu_features__
from optuna.trial import TrialState
from sklearn.base import BaseEstimator
from sklearn.metrics import confusion_matrix, f1_score

The shared modules of the thesis

In [ ]:
from avcore import (
    classifiers,
    data,
    evaluation,
    features,
    progress,
    runs,
    tracking,
    tuning,
)

# Configuration

The vectorizer search space of each representation, every entry suggested in this order

In [ ]:
VECTORIZER_SEARCH_SPACES = {
    "tfidf": {
        "max_features": ("categorical", 20_000, 50_000, 100_000, None),
        "min_df": ("float", 1e-5, 0.01),
        "lowercase": ("categorical", False, True),
        "sublinear_tf": ("categorical", True, False),
    },
    "bow": {
        "max_features": ("categorical", 20_000, 50_000, 100_000, None),
        "min_df": ("float", 1e-5, 0.01),
        "max_df": ("float", 0.7, 1.0),
        "lowercase": ("categorical", False, True),
        "binary": ("categorical", False, True),
    },
}

The function that builds the features of each representation, and its part of the run name

In [ ]:
FEATURE_BUILDERS = {
    "tfidf": features.build_tfidf_features,
    "bow": features.build_bow_features,
}
RUN_NAME_PARTS = {"tfidf": "word_TF-IDF", "bow": "word_BOW"}

The two combinations of the vectors of a pair, and their names in the result tables

In [ ]:
COMBINE_FUNCTIONS = {
    "absolute_difference": features.compute_absolute_difference_features,
    "element-wise": features.compute_element_wise_features,
}
COMBINATION_TABLE_NAMES = {
    "absolute_difference": "absolute_difference",
    "element-wise": "elementwise_product",
}

A parameter outside these lists stops the run here

In [ ]:
if REPRESENTATION not in VECTORIZER_SEARCH_SPACES:
    raise ValueError(f"Unknown REPRESENTATION {REPRESENTATION!r}.")
if COMBINATION not in COMBINE_FUNCTIONS:
    raise ValueError(f"Unknown COMBINATION {COMBINATION!r}.")
if CLASSIFIER not in classifiers.CLASSIFIER_NAMES:
    raise ValueError(f"Unknown CLASSIFIER {CLASSIFIER!r}.")

The settings of this run

In [ ]:
VECTORIZER_SEARCH_SPACE = VECTORIZER_SEARCH_SPACES[REPRESENTATION]
BUILD_FEATURES = FEATURE_BUILDERS[REPRESENTATION]
COMBINE_PAIR_FEATURES = COMBINE_FUNCTIONS[COMBINATION]
CLASSIFIER_SEARCH_SPACE = classifiers.CLASSIFIER_SEARCH_SPACES[CLASSIFIER]
# every lowercase choice of the search space (its first item is the kind), each gets token counts
LOWERCASE_VALUES = VECTORIZER_SEARCH_SPACE["lowercase"][1:]
TRAINING_SECONDS_ATTR = "training_seconds"
DEVELOPER_INITIALS = "JP"

What the run is, in the words of the result tables

In [ ]:
RUN_IDENTITY = {
    "family": "traditional",
    "model": "none",
    "representation": REPRESENTATION,
    "combination": COMBINATION_TABLE_NAMES[COMBINATION],
    "classifier": CLASSIFIER,
}

The smoke run - kNN needs more training pairs than its largest number of neighbours (50)

In [ ]:
SMOKE_ROWS = {"train": 120, "validation": 40, "test": 40}

The data files

In [ ]:
# the notebook runs from its own folder, two levels below the experiments folder
PROJECT_DIR = Path.cwd().parents[1]
DATA_DIR = PROJECT_DIR / "data"
CLEANED_PATHS = {
    "train": DATA_DIR / "processed" / data.TRAIN_CLEANED_FILE,
    "validation": DATA_DIR / "processed" / data.VALIDATION_CLEANED_FILE,
    "test": DATA_DIR / "processed" / data.TEST_CLEANED_FILE,
}